Create Model 1 using Tensorflow

In [ ]:
import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
import pandas as pd
import PIL

from tensorflow import keras
from tensorflow.keras import layers

datasetPath = kagglehub.dataset_download("ongshujian/real-and-fake-pokemon-cards")

def build_tensor_model(hp):
    base_filters = hp.Choice("base_filter", [16, 32]) # for feature extraction
    dense_units = hp.Choice("dense_units", [64, 128]) # for feature classification 
    model = keras.models.Sequential([
        layers.Conv2D(base_filters, 3, padding='same', activation='relu'),
        layers.MaxPooling2D(),
        layers.Conv2D(base_filters * 2, 3, padding='same', activation='relu'),
        layers.MaxPooling2D(),
        layers.Conv2D(base_filters * 4, 3, padding='same', activation='relu'),
        layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dense(dense_units, activation='relu'),
        layers.Dense(2)
    ])
    model.compile(optimizer='adam', loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True), metrics=['accuracy'])
    return model



Using Colab cache for faster access to the 'real-and-fake-pokemon-cards' dataset.


Process the data and split into training and validation sets. Use Tensorflows function to work.

In [ ]:
#install keras tuner so we can tune hypermodels in Tensorflow
%pip install -q -U keras-tuner

import os
#we'll use this tool for training validation split for relative ease of use
from sklearn.model_selection import train_test_split, GridSearchCV
import keras_tuner as kt

df = pd.read_csv(f"{datasetPath}/train_labels.csv") 
#shuffle dataset because the original dataset is ordered. We want to keep 
photoID_training, photoID_valid, classify_training, classify_valid = train_test_split(
    df["id"], df["label"],
    test_size= .2,
    stratify = df["label"],
    random_state=42
)


def processThroughImages(imagesCollection, labels, stage):
    imagesArray = []
    labelToReturn= []
    # Images 1 - 298 in our dataset will be our training data. Rest will be testing
    for imageID, label in zip(imagesCollection, labels):
        imageToProcessFilePath = os.path.join(datasetPath, stage, f"{imageID}.JPG")
        
        if not os.path.exists(imageToProcessFilePath):
            print(f"Missing: {imageToProcessFilePath}")
            continue

        imageToProcess = keras.utils.img_to_array(keras.utils.load_img(imageToProcessFilePath, target_size=[224,224]))
        
        imagesArray.append(imageToProcess)
        labelToReturn.append(label)
    
    
    return np.array(imagesArray), np.array(labelToReturn)

trainingImagesArray, trainingClassifyArray = processThroughImages(photoID_training, classify_training, "train")
validationImagesArray, validationClassifyArray = processThroughImages(photoID_valid, classify_valid, "train")

#normalize for model's ease
trainingImagesArray /= 255.0
validationImagesArray /= 255.0

# hypertune parameters with model 1 and see what we can do
model1Tuner = kt.GridSearch(
    build_tensor_model,
    objective="val_accuracy",
    max_trials=4,
    directory="fakeCardTensorflowLogs",
    project_name="pokemonFakeCardClassifier"
)

model1Tuner.search(
    trainingImagesArray, trainingClassifyArray,
    epochs = 5,
    validation_data=(validationImagesArray, validationClassifyArray)
)

# display our results of our tuning here. 

# rebuild with best hyperparameters and get our final tuned model
model1 = model1Tuner.hypermodel.build(model1Tuner.get_best_hyperparameters(1)[0])
model1.fit(
    trainingImagesArray, trainingClassifyArray, 
    epochs = 5,
    validation_data=(validationImagesArray, validationClassifyArray)
)

model1.summary()



Reloading Tuner from fakeCardTensorflowLogs/pokemonFakeCardClassifier/tuner0.json


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_6 (Conv2D)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_8 (Conv2D)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_8 (MaxPooling2D)  │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
10/10 ━━━━━━━━━━━━━━━━━━━━ 42s 4s/step - accuracy: 0.5772 - loss: 1.0512 - val_accuracy: 0.6667 - val_loss: 0.7723
Epoch 2/5
10/10 ━━━━━━━━━━━━━━━━━━━━ 39s 4s/step - accuracy: 0.6711 - loss: 0.6772 - val_accuracy: 0.6667 - val_loss: 0.6037
Epoch 3/5
10/10 ━━━━━━━━━━━━━━━━━━━━ 40s 4s/step - accuracy: 0.6711 - loss: 0.5823 - val_accuracy: 0.6667 - val_loss: 0.5216
Epoch 4/5
10/10 ━━━━━━━━━━━━━━━━━━━━ 39s 4s/step - accuracy: 0.7215 - loss: 0.5092 - val_accuracy: 0.8400 - val_loss: 0.4742
Epoch 5/5
10/10 ━━━━━━━━━━━━━━━━━━━━ 48s 5s/step - accuracy: 0.7987 - loss: 0.4515 - val_accuracy: 0.8267 - val_loss: 0.4454


Let's try creation a second model and see which one is more accurate. We're going to build a scikit learn shallow learning model.

In [12]:
from sklearn import tree
from sklearn.metrics import accuracy_score, classification_report
# I have to flatten arrays because sklearn only allows 1D arrays
def flattenArrays(arr):
    return arr.reshape(len(arr), -1)

#Let's tune hyperparameters
#tune max_depth
param_grid = {
  #add params to test
  'max_depth': [None, *range(1, 25)],
  'max_features': [1, 2, 3, 4]
}

grid_search = GridSearchCV(tree.DecisionTreeClassifier(), param_grid=param_grid, scoring='accuracy')

grid_search.fit(flattenArrays(trainingImagesArray), flattenArrays(trainingClassifyArray))
print(grid_search.best_params_)

model2 = grid_search.best_estimator_
validationCheck = model2.predict(flattenArrays(validationImagesArray))
print(accuracy_score(validationCheck, validationClassifyArray))
print(classification_report(validationCheck, validationClassifyArray))

{'max_depth': 24, 'max_features': 4}
1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        25
           1       1.00      1.00      1.00        50

    accuracy                           1.00        75
   macro avg       1.00      1.00      1.00        75
weighted avg       1.00      1.00      1.00        75



Test it on the test data we have set, then compare the two models. Pick the best one from here and export as a backend api

In [14]:
#prepare test set
testDF = pd.read_csv(f"{datasetPath}/test_labels.csv").sample(frac=1, random_state=42).reset_index(drop=True)

# test model 2
testImagesArray, testClassifyArray = processThroughImages(testDF["id"], testDF["label"], "test")
testCheck = model2.predict(flattenArrays(testImagesArray))
print(accuracy_score(testCheck, testClassifyArray))
print(classification_report(testCheck, testClassifyArray))


TypeError: processThroughImages() takes 2 positional arguments but 3 were given